This notebook produces two drought hazard indices per census tract following VCP's three-indicator methodology:

- **`drought_hazard_idx_norm`** (current): equal-weight mean of normalized June–August temperature change (current), Water Shortage Vulnerability, and precipitation/demand ratio
- **`drought_hazard_fut_idx_norm`** (mid-century, 2041–2070): same, with mid-century June–August temperature change

June–August temperature change is normalized min-max across both periods combined (VCP methodology). Water Shortage Vulnerability and precipitation/demand ratio are the same value in both timescales per VCP and are normalized once. Both indices are normalized to 0–100.

Cal-Adapt SPEI-12 and SPEI-36 mid-century drought frequency are included as standalone variables. These capture drought incidence directly but are only available for mid-century (no historic equivalent — the historic baseline is ~5% by construction of the 5th-percentile threshold), so they are not included in the composite.

In [ ]:
import json
import pandas as pd
import numpy as np

## Import Data

In [ ]:
# VCP drought indicators
with open("data_sources/hazards/VCP_Tracts.geojson") as f:
    vcp_data = json.load(f)

vcp = pd.DataFrame([
    {
        "GEOID": str(feat["properties"]["GEOID"]),
        "Dr_delta_JA_max_pre":    feat["properties"].get("Dr_delta_JA_max_pre"),
        "Dr_delta_JA_max_fut":    feat["properties"].get("Dr_delta_JA_max_fut"),
        "Dr_WSV_average":         feat["properties"].get("Dr_WSV_average"),
        "Dr_precip_demand_ratio": feat["properties"].get("Dr_precip_demand_ratio"),
    }
    for feat in vcp_data["features"]
])
print(f"{len(vcp)} VCP tracts loaded")
print(vcp.isnull().sum())

In [ ]:
# Cal-Adapt SPEI-12 delta (mid-century only, standalone)
spei = pd.read_csv(
    "data_sources/hazards/drought/droughtfrequency_tract.csv",
    dtype={"GEOID": str}
)[["GEOID", "drought_delta_spei12_midcentury"]]
print(f"{len(spei)} Cal-Adapt tracts loaded")

In [ ]:
df = pd.merge(vcp, spei, on="GEOID", how="left")
print(f"{len(df)} tracts after merge")

## Normalize

In [ ]:
# Normalize June–August temperature change across both periods combined (VCP methodology)
ja_all = pd.concat([df["Dr_delta_JA_max_pre"], df["Dr_delta_JA_max_fut"]], ignore_index=True)
ja_min, ja_max = ja_all.min(), ja_all.max()
print(f"JA temp change range (both periods): {ja_min:.3f} – {ja_max:.3f}")

df["ja_pre_norm"] = ((df["Dr_delta_JA_max_pre"] - ja_min) / (ja_max - ja_min)) * 100
df["ja_fut_norm"] = ((df["Dr_delta_JA_max_fut"] - ja_min) / (ja_max - ja_min)) * 100

In [ ]:
# Normalize WSV and precip/demand ratio (single timescale each)
# precip/demand ratio is inverted: higher ratio = more precipitation relative to demand
# = lower drought risk, so it must score low in the drought index.
def norm_0_100(series):
    s_min, s_max = series.min(), series.max()
    return ((series - s_min) / (s_max - s_min)) * 100

df["wsv_norm"]           = norm_0_100(df["Dr_WSV_average"])
df["precip_demand_norm"] = 100 - norm_0_100(df["Dr_precip_demand_ratio"])

## Create Indices

Three equal-weight components following VCP's drought methodology: June–August temperature change (evapotranspiration demand), Water Shortage Vulnerability (infrastructure/groundwater risk), and precipitation/demand ratio (supply-demand balance).

In [ ]:
df["drought_hazard_raw"]     = df[["ja_pre_norm", "wsv_norm", "precip_demand_norm"]].mean(axis=1)
df["drought_hazard_fut_raw"] = df[["ja_fut_norm", "wsv_norm", "precip_demand_norm"]].mean(axis=1)

df["drought_hazard_idx_norm"]     = norm_0_100(df["drought_hazard_raw"])
df["drought_hazard_fut_idx_norm"] = norm_0_100(df["drought_hazard_fut_raw"])

df[["drought_hazard_idx_norm", "drought_hazard_fut_idx_norm"]].describe()

## Export

In [ ]:
output = df[[
    "GEOID",
    "Dr_delta_JA_max_pre",
    "Dr_delta_JA_max_fut",
    "Dr_WSV_average",
    "Dr_precip_demand_ratio",
    "drought_delta_spei12_midcentury",
    "drought_hazard_idx_norm",
    "drought_hazard_fut_idx_norm",
]]
output.to_csv("data/drought_hazard.csv", index=False)
print(f"Wrote {len(output)} rows → data/drought_hazard.csv")
output.head()